# Deteksi Sampah dari Video CCTV — YOLO26s (versi perbaikan)

Perbaikan utama dari notebook sebelumnya:

1. **Bug konversi TACO diperbaiki.** Bbox TACO ada di koordinat gambar *asli*, tetapi sebelumnya dinormalisasi dengan ukuran gambar 640px hasil download, jadi banyak box salah posisi / terbuang (901 gambar → hanya 294 tersimpan). Sekarang dinormalisasi dengan `width/height` dari anotasi + orientasi EXIF dicek.
2. `Plastic bottle cap` tidak lagi masuk kelas botol (tutup botol ≠ botol).
3. **Kelas disesuaikan dengan dashboard TrashSpot**: `botol_plastik`, `kantong_plastik`, `kaleng`, `kardus` (pemetaan dari kategori TACO dicetak di sel dataset).
4. Evaluasi **sebelum vs sesudah fine-tune** (TACO val dan CCTV val terpisah), model final dipilih otomatis.
5. Mendukung **banyak video CCTV** sekaligus + pengecekan apakah label benar-benar sudah dikoreksi.
6. Fungsi deteksi dipisah jadi **`detector.py`** (bisa langsung dipakai Streamlit), bukan lagi fungsi di dalam notebook.

> Runtime: **GPU (T4)**. Jalankan sel dari atas ke bawah.
7. **Status kebersihan 4 tingkat** (Bersih, Cukup Bersih, Perlu Dibersihkan, Sangat Perlu Dibersihkan) + modul **`dashboard_data.py`** untuk riwayat pemeriksaan, ringkasan desa, dan prioritas pembersihan.

> Karena kelas berubah, model **harus dilatih ulang** (bobot lama `.pt` tidak cocok).

8. **Hitung sampah menetap tahan kamera goyang**: gerak kamera dikompensasi (phase correlation) dan pergeseran diukur dengan simpangan baku, bukan selisih ekstrem. Sebelumnya semua sampah dianggap bergerak sehingga lokasi selalu "Bersih".
9. Fine-tune otomatis **dilewati** kalau label CCTV belum dikoreksi (supaya tidak belajar dari tebakannya sendiri).


In [2]:
!pip -q install -U "ultralytics>=8.4.0" lap
!pip -q show ultralytics | grep -i ^version

In [3]:
import os, re, glob, shutil, zipfile, random, json, time, csv, subprocess, sys
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import cv2, numpy as np, torch, yaml, requests
from PIL import Image, ImageOps
from google.colab import files
from IPython.display import display, Video

# ===================== KONFIGURASI =====================
MODEL_NAME   = "yolo26s.pt"
EPOCHS       = 60               # training dari TACO (early stopping aktif)
FT_EPOCHS    = 40               # fine-tune TACO + CCTV
IMGSZ        = 640
BATCH        = 16
PATIENCE     = 30
SEED         = 42

# Ekstraksi frame CCTV (per video)
FRAME_EVERY_SEC = 0.5
MIN_DIFF        = 1.0           # makin besar = makin sedikit frame (buang frame mirip)
MAX_FRAMES      = 150           # maksimum frame per video
MAX_SIDE        = 1280

# Pseudo-label & fine-tune
PSEUDO_CONF  = 0.20             # model masih lemah -> threshold rendah, nanti dikoreksi manual
CCTV_REPEAT  = 5                # frame CCTV diulang N kali di train (jumlahnya kecil vs TACO)

# Deteksi + tracking
CONF         = 0.25
TRACKER      = "bytetrack.yaml"
MIN_HITS     = 5
STATIC_SEC   = 2.0
STATIC_MOVE  = 0.06           # sampah dianggap diam kalau sebarannya < 6% diagonal frame (setelah kompensasi kamera)
COMPENSATE_CAMERA = True        # buang gerak kamera (goyang/pan) sebelum menghitung pergeseran
OUT_WIDTH    = 1280

# ----- Status kebersihan (sesuai dashboard) -----
HITUNG_DARI      = "menetap"        # "menetap" = hanya sampah diam (orang lewat bawa botol tidak dihitung)
                                    # "unik"    = semua sampah yang pernah terdeteksi
BATAS_KEBERSIHAN = (5, 15, 28)      # jumlah sampah: <=5 Bersih | <=15 Cukup Bersih
                                    #               <=28 Perlu Dibersihkan | >28 Sangat Perlu Dibersihkan

# ----- Lokasi yang sedang dipantau (muncul di dashboard) -----
LOKASI = "Taman"
BANJAR = "Banjar Contoh"
KAMERA = "CCTV 1"
DASH_DIR     = "/content/dashboard_data"
RIWAYAT_JSON = f"{DASH_DIR}/riwayat.json"

BASE      = "/content"
TACO_SRC  = Path(f"{BASE}/TACO")
TACO_RAW  = Path(f"{BASE}/taco_selected_images")
TACO_DIR  = Path(f"{BASE}/taco_yolo")
CCTV_DIR  = Path(f"{BASE}/cctv_yolo")
RAW_DIR   = Path(f"{BASE}/cctv_frames")
RUNS_DIR  = f"{BASE}/runs/detect"
RUN_NAME  = "yolo26s_taco"
FT_NAME   = "yolo26s_taco_cctv"
DRIVE_OUT = "/content/drive/MyDrive/taco_yolo"

CLASS_NAMES = ["botol_plastik", "kantong_plastik", "kaleng", "kardus"]   # sesuai dashboard TrashSpot
DEVICE = 0 if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED)
print("Device:", DEVICE, "| Model:", MODEL_NAME, "| Kelas:", CLASS_NAMES)
if DEVICE == "cpu":
    print("PERINGATAN: GPU tidak aktif. Runtime > Change runtime type > T4 GPU.")

# Google Drive (opsional) agar bobot & dataset tidak hilang saat runtime putus
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT, exist_ok=True)
except Exception as e:
    print("Drive tidak di-mount:", repr(e))

Device: 0 | Model: yolo26s.pt | Kelas: ['botol_plastik', 'kantong_plastik', 'kaleng', 'kardus']
Mounted at /content/drive


## 1. Dataset TACO

In [4]:
if not TACO_SRC.exists():
    !git clone --depth 1 https://github.com/pedropro/TACO.git {TACO_SRC}
else:
    print("Repo TACO sudah ada:", TACO_SRC)

Cloning into '/content/TACO'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 31 (delta 2), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 22.79 MiB | 18.16 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [5]:
REBUILD_TACO = True    # WAJIB True saat kelas berubah (gambar yang sudah diunduh dipakai ulang). Boleh False kalau dataset sudah dibangun dengan kelas ini.

def map_category(category_name):
    """Nama kategori TACO -> indeks kelas kita (None = diabaikan)."""
    n = category_name.lower().strip()
    if re.search(r"\bcap\b|\blid\b", n):                  # tutup botol/kaleng bukan botol/kaleng
        return None
    if "plastic bottle" in n:
        key = "botol_plastik"
    elif n in {"drink can", "food can", "can"}:
        key = "kaleng"
    elif any(t in n for t in ["carrier bag", "polypropylene bag", "garbage bag", "plastic film", "plastic wrapper"]):
        key = "kantong_plastik"
    elif "carton" in n or "pizza box" in n:
        key = "kardus"
    else:
        return None
    return CLASS_NAMES.index(key)


def build_taco():
    ann_file = TACO_SRC / "data" / "annotations.json"
    if not ann_file.exists():
        raise FileNotFoundError(f"Anotasi tidak ditemukan: {ann_file}")
    coco = json.loads(ann_file.read_text(encoding="utf-8"))

    category_map = {}
    print("=== KATEGORI TACO YANG DIPETAKAN ===")
    for cat in coco["categories"]:
        m = map_category(cat["name"])
        if m is not None:
            category_map[cat["id"]] = m
            print(f'  {cat["name"]} -> {CLASS_NAMES[m]}')
    if not category_map:
        raise ValueError("Tidak ada kategori yang cocok.")

    images_by_id = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        bbox = ann.get("bbox")
        if (ann.get("category_id") in category_map and ann.get("image_id") in images_by_id
                and bbox and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0):
            anns_by_image[ann["image_id"]].append(ann)
    print("\nGambar dengan anotasi terpilih:", len(anns_by_image))

    shutil.rmtree(TACO_DIR, ignore_errors=True)
    TACO_RAW.mkdir(parents=True, exist_ok=True)         # cache unduhan dipertahankan
    for split in ("train", "val"):
        (TACO_DIR / "images" / split).mkdir(parents=True)
        (TACO_DIR / "labels" / split).mkdir(parents=True)

    headers = {"User-Agent": "Mozilla/5.0"}

    def download_one(image_id):
        info = images_by_id[image_id]
        out = TACO_RAW / f"image_{image_id}.jpg"
        if out.exists() and out.stat().st_size > 0:
            try:
                with Image.open(out) as im:
                    im.verify()
                return image_id, out
            except Exception:
                pass
        for url in [info.get("flickr_640_url"), info.get("flickr_url")]:
            if not url:
                continue
            for _ in range(2):
                try:
                    r = requests.get(url, timeout=30, headers=headers)
                    r.raise_for_status()
                    if not r.headers.get("Content-Type", "").lower().startswith("image/") or not r.content:
                        continue
                    out.write_bytes(r.content)
                    with Image.open(out) as im:
                        im.verify()
                    return image_id, out
                except Exception:
                    time.sleep(0.5)
        return image_id, None

    ids = sorted(anns_by_image)
    downloaded, failed = {}, []
    with ThreadPoolExecutor(max_workers=8) as ex:
        for i, (iid, path) in enumerate(ex.map(download_one, ids), 1):
            (downloaded.__setitem__(iid, path) if path else failed.append(iid))
            if i % 100 == 0 or i == len(ids):
                print(f"Download: {i}/{len(ids)}")
    print("Berhasil:", len(downloaded), "| Gagal:", len(failed))
    if len(downloaded) < 5:
        raise RuntimeError("Gambar yang berhasil diunduh terlalu sedikit. Periksa koneksi.")

    image_ids = sorted(downloaded)
    random.shuffle(image_ids)
    sp = max(1, min(int(len(image_ids) * 0.8), len(image_ids) - 1))
    splits = {"train": image_ids[:sp], "val": image_ids[sp:]}

    # ---- COCO -> YOLO ----
    # PENTING: bbox TACO ada di koordinat gambar ASLI (info["width"] x info["height"]).
    # Gambar hasil download bisa berukuran lain (640px) -> normalisasi pakai ukuran anotasi.
    saved, skip_aspect, skip_box = 0, 0, 0
    for split, split_ids in splits.items():
        for iid in split_ids:
            info = images_by_id[iid]
            try:
                raw = Image.open(downloaded[iid]); raw.load()
            except Exception:
                continue
            aw, ah = info.get("width") or raw.size[0], info.get("height") or raw.size[1]
            img = None
            for cand in (ImageOps.exif_transpose(raw), raw):       # cocokkan orientasi dgn anotasi
                if abs(cand.size[0] / cand.size[1] - aw / ah) < 0.05:
                    img = cand; break
            if img is None:
                skip_aspect += 1
                continue

            lines = []
            for ann in anns_by_image[iid]:
                x, y, w, h = ann["bbox"]
                x1, y1 = max(0.0, min(aw, x)), max(0.0, min(ah, y))
                x2, y2 = max(0.0, min(aw, x + w)), max(0.0, min(ah, y + h))
                bw, bh = x2 - x1, y2 - y1
                if bw < 2 or bh < 2:
                    skip_box += 1
                    continue
                lines.append(f"{category_map[ann['category_id']]} {(x1+x2)/2/aw:.6f} {(y1+y2)/2/ah:.6f} {bw/aw:.6f} {bh/ah:.6f}")
            if lines:
                img.convert("RGB").save(TACO_DIR / "images" / split / f"image_{iid}.jpg", quality=95)
                (TACO_DIR / "labels" / split / f"image_{iid}.txt").write_text("\n".join(lines), encoding="utf-8")
                saved += 1

    yaml_data = {"path": str(TACO_DIR), "train": "images/train", "val": "images/val",
                 "names": {i: n for i, n in enumerate(CLASS_NAMES)}}
    (TACO_DIR / "data.yaml").write_text(yaml.safe_dump(yaml_data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"\n=== DATASET SELESAI === tersimpan: {saved} | dilewati (orientasi tak cocok): {skip_aspect} | box terlalu kecil: {skip_box}")


if REBUILD_TACO or not (TACO_DIR / "data.yaml").exists():
    build_taco()
else:
    print("Dataset TACO sudah ada, dilewati:", TACO_DIR)

DATA_YAML = str(TACO_DIR / "data.yaml")

=== KATEGORI TACO YANG DIPETAKAN ===
  Other plastic bottle -> botol_plastik
  Clear plastic bottle -> botol_plastik
  Food Can -> kaleng
  Drink can -> kaleng
  Other carton -> kardus
  Egg carton -> kardus
  Drink carton -> kardus
  Corrugated carton -> kardus
  Meal carton -> kardus
  Pizza box -> kardus
  Plastic film -> kantong_plastik
  Garbage bag -> kantong_plastik
  Other plastic wrapper -> kantong_plastik
  Single-use carrier bag -> kantong_plastik
  Polypropylene bag -> kantong_plastik

Gambar dengan anotasi terpilih: 923
Download: 100/923
Download: 200/923
Download: 300/923
Download: 400/923
Download: 500/923
Download: 600/923
Download: 700/923
Download: 800/923
Download: 900/923
Download: 923/923
Berhasil: 923 | Gagal: 0

=== DATASET SELESAI === tersimpan: 923 | dilewati (orientasi tak cocok): 0 | box terlalu kecil: 0


In [6]:
# Cek distribusi anotasi per kelas
for split in ("train", "val"):
    counts = Counter()
    for f in (TACO_DIR / "labels" / split).glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    n_img = len(list((TACO_DIR / "images" / split).glob("*")))
    print(f"\n[{split}] {n_img} gambar")
    for i, n in enumerate(CLASS_NAMES):
        print(f"  {n:<16}: {counts[i]} anotasi")
        if counts[i] < 30:
            print(f"    PERINGATAN: kelas '{n}' sangat sedikit, AP kelas ini tidak bisa diandalkan.")


[train] 738 gambar
  botol_plastik   : 275 anotasi
  kantong_plastik : 662 anotasi
  kaleng          : 196 anotasi
  kardus          : 188 anotasi

[val] 185 gambar
  botol_plastik   : 60 anotasi
  kantong_plastik : 144 anotasi
  kaleng          : 67 anotasi
  kardus          : 58 anotasi


## 2. Training YOLO26s dari TACO

In [7]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=2,
    project=RUNS_DIR,
    name=RUN_NAME,
    exist_ok=True,
    seed=SEED,
    patience=PATIENCE,
    cos_lr=True,
    plots=True,
    amp=True,
    verbose=True,
)

BEST_TACO = str(model.trainer.best)
print("Model ditemukan:", os.path.exists(BEST_TACO), "|", BEST_TACO)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/taco_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fre

In [8]:
def eval_model(weights, data, name, verbose_cls=False):
    """Return (mAP50, mAP50-95). Aman kalau val kosong."""
    try:
        r = YOLO(weights).val(data=data, imgsz=IMGSZ, device=DEVICE, plots=False, verbose=False,
                              project=RUNS_DIR, name=f"val_{name}", exist_ok=True)
        if verbose_cls:
            print(f"P: {r.box.mp:.3f} | R: {r.box.mr:.3f}")
            for i, c in enumerate(r.box.ap_class_index):
                print(f"  {r.names[int(c)]:<16} AP50={r.box.ap50[i]:.3f}  AP50-95={r.box.ap[i]:.3f}")
        return float(r.box.map50), float(r.box.map)
    except Exception as e:
        print("  evaluasi gagal:", repr(e))
        return 0.0, 0.0

m50, m5095 = eval_model(BEST_TACO, DATA_YAML, "taco", verbose_cls=True)
print(f"\nTACO val -> mAP50: {m50:.3f} | mAP50-95: {m5095:.3f}")

try:
    shutil.copy2(BEST_TACO, f"{DRIVE_OUT}/yolo26s_taco_best.pt")
    print("Tersimpan di Drive:", f"{DRIVE_OUT}/yolo26s_taco_best.pt")
except Exception as e:
    print("Backup ke Drive dilewati:", repr(e))

Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 120 layers, 9,466,728 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2820.3±1611.4 MB/s, size: 1312.1 KB)
val: Scanning /content/taco_yolo/labels/val.cache... 185 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 185/185 59.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.4it/s 8.4s
                   all        185        329      0.658      0.465      0.502      0.398
Speed: 1.4ms preprocess, 8.3ms inference, 0.0ms loss, 2.3ms postprocess per image
P: 0.658 | R: 0.465
  botol_plastik    AP50=0.625  AP50-95=0.494
  kantong_plastik  AP50=0.400  AP50-95=0.285
  kaleng           AP50=0.560  AP50-95=0.464
  kardus           AP50=0.423  AP50-95=0.350

TACO val -> mAP50: 0.502 | mAP50-95: 0.398
Tersimpan di Drive: /content/drive/MyDrive/taco_yolo/yolo26s_taco_best.

## 3. Modul deteksi `detector.py`

Sel ini menulis `detector.py` (deteksi + tracking + hitung sampah unik/menetap). File ini yang nanti dipakai langsung oleh dashboard Streamlit.

Sel berikutnya menulis `dashboard_data.py` (riwayat pemeriksaan, ringkasan desa, prioritas pembersihan) yang membaca hasil dari `detector.py`.


In [9]:
%%writefile /content/detector.py
"""detector.py - deteksi + tracking sampah dari video CCTV (YOLO + ByteTrack).

Pemakaian:
    from detector import analyze_video
    hasil = analyze_video("video.mp4", "best.pt", out_dir="output")
    hasil["summary"], hasil["tracks"], hasil["video"], hasil["csv"]
"""
import csv
import json
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

PALETTE = [(0, 170, 255), (80, 220, 80), (255, 140, 40), (230, 80, 200), (80, 200, 255)]
FONT = cv2.FONT_HERSHEY_SIMPLEX

# Tingkat kebersihan sesuai dashboard (urut dari terbersih ke terkotor)
STATUS = {
    "bersih":                   {"label": "Bersih",                   "hex": "#1F7A5C", "bgr": (92, 122, 31),  "prioritas": "Rendah"},
    "cukup_bersih":             {"label": "Cukup Bersih",             "hex": "#8DA52E", "bgr": (46, 165, 141), "prioritas": "Rendah"},
    "perlu_dibersihkan":        {"label": "Perlu Dibersihkan",        "hex": "#E0A21B", "bgr": (27, 162, 224), "prioritas": "Sedang"},
    "sangat_perlu_dibersihkan": {"label": "Sangat Perlu Dibersihkan", "hex": "#C2410C", "bgr": (12, 65, 194),  "prioritas": "Tinggi"},
}
STATUS_URUT = list(STATUS)

LABEL_JENIS = {"botol_plastik": "Botol plastik", "kantong_plastik": "Kantong plastik",
               "kaleng": "Kaleng", "kardus": "Kardus"}


def label_jenis(nama):
    return LABEL_JENIS.get(nama, nama.replace("_", " ").capitalize())


def klasifikasi_kebersihan(jumlah, batas=(5, 15, 28)):
    """Jumlah sampah -> status kebersihan. batas = (maks Bersih, maks Cukup Bersih, maks Perlu Dibersihkan)."""
    b1, b2, b3 = batas
    if jumlah <= b1:
        kode, info = "bersih", f"Lokasi bersih. Hanya {jumlah} sampah terdeteksi."
    elif jumlah <= b2:
        kode, info = "cukup_bersih", f"Lokasi cukup bersih. Ada {jumlah} sampah, masih aman tetapi perlu dipantau."
    elif jumlah <= b3:
        kode, info = "perlu_dibersihkan", f"Lokasi perlu dibersihkan. Ada {jumlah} sampah, jadwalkan pembersihan."
    else:
        kode, info = "sangat_perlu_dibersihkan", f"Lokasi sangat perlu dibersihkan. Ada {jumlah} sampah, segera tangani."
    s = STATUS[kode]
    return {"kode": kode, "label": s["label"], "prioritas": s["prioritas"], "jumlah_sampah": int(jumlah),
            "keterangan": info, "warna_hex": s["hex"], "tingkat": STATUS_URUT.index(kode), "batas": list(batas)}


def analyze_video(video_path, model, out_dir="output", tag=None, conf=0.25, imgsz=640,
                  tracker="bytetrack.yaml", min_hits=5, static_sec=2.0, static_move=0.06,
                  out_width=1280, device=None, save_video=True, progress=None,
                  hitung_dari="menetap", batas=(5, 15, 28), compensate_camera=True):
    """Deteksi + tracking satu video.

    model         : path .pt atau objek YOLO
    min_hits      : track dihitung valid kalau muncul >= N frame (buang flicker)
    static_sec    : dianggap menetap kalau terlihat >= N detik ...
    static_move   : ... dan bergeser < fraksi ini dari diagonal frame
    save_video    : False = hanya hitung (lebih cepat), tanpa video hasil
    progress      : callback progress(frame_ke, total_frame), mis. untuk st.progress
    hitung_dari   : "menetap" (sampah diam saja) atau "unik" (semua sampah) -> dasar penentuan status
    compensate_camera : True = gerak kamera (goyang/pan) dibuang dulu sebelum menilai apakah sampah diam
    batas         : (maks Bersih, maks Cukup Bersih, maks Perlu Dibersihkan); di atasnya = Sangat Perlu Dibersihkan

    Return dict: video, csv, json, fps, frames, duration_sec, summary (ada "status" & "jumlah_per_jenis"), tracks
    """
    video_path = str(video_path)
    tag = tag or Path(video_path).stem
    os.makedirs(out_dir, exist_ok=True)
    mdl = model if isinstance(model, YOLO) else YOLO(str(model))
    names = mdl.names

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Video tidak bisa dibuka: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    sc = min(1.0, out_width / W)
    ow, oh = int(W * sc) // 2 * 2, int(H * sc) // 2 * 2
    diag = float(np.hypot(W, H))
    fs = max(0.45, ow / 1280 * 0.6)
    th = max(1, round(ow / 640))

    raw_out = os.path.join(out_dir, f"{tag}_raw.mp4")
    writer = cv2.VideoWriter(raw_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh)) if save_video else None

    tracks = {}

    def extent(t):
        """Sebaran posisi track (4 x simpangan baku, gerak kamera sudah dibuang) dibagi diagonal frame."""
        n = t["n"]
        vx = max(0.0, t["sxx"] / n - (t["sx"] / n) ** 2)
        vy = max(0.0, t["syy"] / n - (t["sy"] / n) ** 2)
        return 4.0 * float(np.sqrt(vx + vy)) / diag

    def is_static(t):
        span = (t["last"] - t["first"] + 1) / fps
        return span >= static_sec and extent(t) <= static_move

    def tally():
        uniq, stat = Counter(), Counter()
        for t in tracks.values():
            if t["n"] >= min_hits:
                c = t["votes"].most_common(1)[0][0]
                uniq[c] += 1
                stat[c] += int(is_static(t))
        return uniq, stat

    # estimasi gerak kamera antar frame (translasi global) -> posisi sampah dihitung di "koordinat dunia"
    gw = 320
    gh = max(2, int(round(gw * H / W)))
    hann = cv2.createHanningWindow((gw, gh), cv2.CV_32F)
    prev_g, prev_textured, cam_x, cam_y = None, False, 0.0, 0.0

    fi = -1
    for fi, r in enumerate(mdl.track(source=video_path, stream=True, conf=conf, imgsz=imgsz,
                                     device=device, tracker=tracker, verbose=False)):
        if compensate_camera:
            g = cv2.cvtColor(cv2.resize(r.orig_img, (gw, gh), interpolation=cv2.INTER_AREA),
                             cv2.COLOR_BGR2GRAY).astype(np.float32)
            textured = float(g.std()) >= 3.0           # frame polos/gelap -> phase correlation memberi geseran palsu
            if prev_g is not None and textured and prev_textured:
                (dx, dy), resp = cv2.phaseCorrelate(prev_g, g, hann)
                if resp > 0.1 and abs(dx) < gw * 0.25 and abs(dy) < gh * 0.25:
                    cam_x -= dx * W / gw
                    cam_y -= dy * H / gh
            prev_g, prev_textured = g, textured

        frame = cv2.resize(r.orig_img, (ow, oh), interpolation=cv2.INTER_AREA) if save_video else None

        if r.boxes is not None and r.boxes.id is not None:
            ids = r.boxes.id.int().cpu().tolist()
            cls = r.boxes.cls.int().cpu().tolist()
            cfs = r.boxes.conf.cpu().tolist()
            xyxy = r.boxes.xyxy.cpu().numpy()
            for tid, c, cf, b in zip(ids, cls, cfs, xyxy):
                cx, cy = (b[0] + b[2]) / 2 - cam_x, (b[1] + b[3]) / 2 - cam_y
                t = tracks.get(tid)
                if t is None:
                    t = tracks[tid] = dict(first=fi, last=fi, n=0, votes=Counter(), confsum=0.0,
                                           sx=0.0, sy=0.0, sxx=0.0, syy=0.0)
                t["last"] = fi; t["n"] += 1; t["votes"][c] += 1; t["confsum"] += cf
                t["sx"] += cx; t["sy"] += cy; t["sxx"] += cx * cx; t["syy"] += cy * cy

                if frame is None or t["n"] < min_hits:
                    continue
                static = is_static(t)
                col = (0, 0, 255) if static else PALETTE[c % len(PALETTE)]
                x1, y1, x2, y2 = [int(v * sc) for v in b]
                cv2.rectangle(frame, (x1, y1), (x2, y2), col, th + (1 if static else 0))
                label = f"#{tid} {names[c]} {cf:.2f}" + (" MENETAP" if static else "")
                (tw, thh), bl = cv2.getTextSize(label, FONT, fs, 1)
                ty = max(y1, thh + bl + 2)
                cv2.rectangle(frame, (x1, ty - thh - bl - 2), (x1 + tw + 4, ty + 2), col, -1)
                cv2.putText(frame, label, (x1 + 2, ty - bl), FONT, fs, (255, 255, 255), 1, cv2.LINE_AA)

        if frame is not None:
            uniq, stat = tally()
            n_basis = sum(stat.values()) if hitung_dari == "menetap" else sum(uniq.values())
            st = klasifikasi_kebersihan(n_basis, batas)
            rows = [f"STATUS: {st['label'].upper()}",
                    f"Sampah unik   : {sum(uniq.values())}", f"Sampah menetap: {sum(stat.values())}"]
            rows += [f"  {label_jenis(names[i])}: {uniq[i]} ({stat[i]} menetap)" for i in sorted(names)]
            lh = int(26 * fs / 0.6 * 0.6) + 8
            pw = int(max(cv2.getTextSize(s, FONT, fs, 1)[0][0] for s in rows)) + 20
            ph = lh * len(rows) + 12
            overlay = frame.copy()
            cv2.rectangle(overlay, (8, 8), (8 + pw, 8 + ph), (0, 0, 0), -1)
            frame = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)
            cv2.rectangle(frame, (8, 8), (8 + pw, 8 + lh + 6), STATUS[st["kode"]]["bgr"], -1)   # bar status berwarna
            for k, s in enumerate(rows):
                cv2.putText(frame, s, (18, 8 + 6 + lh * (k + 1) - 6), FONT, fs, (255, 255, 255), 1, cv2.LINE_AA)
            writer.write(frame)

        if progress and (fi + 1) % 10 == 0:
            progress(fi + 1, total)
    if writer is not None:
        writer.release()
    if progress:
        progress(total, total)

    final = None
    if save_video:
        final = os.path.join(out_dir, f"{tag}_hasil.mp4")
        if shutil.which("ffmpeg"):          # H.264 supaya bisa diputar di browser / Streamlit
            subprocess.run(["ffmpeg", "-y", "-i", raw_out, "-c:v", "libx264", "-preset", "fast",
                            "-crf", "26", "-pix_fmt", "yuv420p", "-an", final],
                           check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            os.remove(raw_out)
        else:
            os.replace(raw_out, final)

    uniq, stat = tally()
    per_kelas = {names[i]: {"unik": int(uniq[i]), "menetap": int(stat[i])} for i in sorted(names)}
    counts = stat if hitung_dari == "menetap" else uniq
    status = klasifikasi_kebersihan(int(sum(counts.values())), batas)
    status["dihitung_dari"] = hitung_dari
    summary = {"total_unik": int(sum(uniq.values())), "total_menetap": int(sum(stat.values())),
               "per_kelas": per_kelas,
               "jumlah_per_jenis": {label_jenis(names[i]): int(counts[i]) for i in sorted(names)},
               "status": status}

    rows = []
    for tid, t in sorted(tracks.items()):
        if t["n"] < min_hits:
            continue
        rows.append({
            "track_id": int(tid),
            "kelas": names[t["votes"].most_common(1)[0][0]],
            "frame_awal": int(t["first"]),
            "frame_akhir": int(t["last"]),
            "muncul_dtk": round(t["first"] / fps, 2),
            "hilang_dtk": round(t["last"] / fps, 2),
            "durasi_dtk": round((t["last"] - t["first"] + 1) / fps, 2),
            "bergeser_pct": round(extent(t) * 100, 2),
            "conf_rata2": round(t["confsum"] / t["n"], 3),
            "menetap": int(is_static(t)),
        })

    csv_path = os.path.join(out_dir, f"{tag}_tracking.csv")
    cols = ["track_id", "kelas", "frame_awal", "frame_akhir", "muncul_dtk", "hilang_dtk",
            "durasi_dtk", "bergeser_pct", "conf_rata2", "menetap"]
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)

    json_path = os.path.join(out_dir, f"{tag}_ringkasan.json")
    hasil = {"video": final, "csv": csv_path, "json": json_path, "fps": float(fps), "frames": fi + 1,
             "duration_sec": round((fi + 1) / fps, 2), "summary": summary, "tracks": rows}
    with open(json_path, "w") as f:
        json.dump(hasil, f, indent=2, ensure_ascii=False)
    return hasil


def print_summary(hasil):
    s = hasil["summary"]
    print(f"{'Kelas':<18}{'Unik':>6}{'Menetap':>10}")
    for k, v in s["per_kelas"].items():
        print(f"{k:<18}{v['unik']:>6}{v['menetap']:>10}")
    print(f"{'TOTAL':<18}{s['total_unik']:>6}{s['total_menetap']:>10}")
    st = s["status"]
    print(f"\nKONDISI KEBERSIHAN: {st['label'].upper()} | prioritas {st['prioritas']} "
          f"| {st['jumlah_sampah']} sampah ({st['dihitung_dari']})")
    print(st["keterangan"])

Writing /content/detector.py


## 3b. Modul `dashboard_data.py`

Riwayat pemeriksaan per lokasi, ringkasan desa, prioritas pembersihan, dan kartu status ala halaman Pantau CCTV.

In [10]:
%%writefile /content/dashboard_data.py
"""dashboard_data.py - riwayat pemeriksaan + ringkasan untuk dashboard TrashSpot.

Alur:
    hasil = analyze_video(...)                                   # detector.py
    catat_pemeriksaan(hasil, lokasi="Taman", banjar="Banjar X", kamera="CCTV 1", path="riwayat.json")
    dash = ringkasan_dashboard(muat_riwayat("riwayat.json"))     # halaman Dashboard
    rwy  = riwayat_lokasi(muat_riwayat("riwayat.json"), "Taman") # halaman Riwayat
    kartu_html(hasil, lokasi="Taman", kamera="CCTV 1")           # panel kanan halaman Pantau CCTV
"""
import json
import os
from collections import Counter
from datetime import datetime

import cv2

from detector import STATUS, STATUS_URUT, klasifikasi_kebersihan, label_jenis

FORMAT_WAKTU = "%Y-%m-%d %H:%M"


# ---------------------------------------------------------------- penyimpanan
def muat_riwayat(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _simpan(path, data):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def simpan_thumbnail(video_path, out_path, posisi=0.5, lebar=320):
    """Ambil 1 frame dari video (default tengah video) sebagai thumbnail riwayat."""
    cap = cv2.VideoCapture(str(video_path))
    ok, frame = False, None
    try:
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * posisi))
        ok, frame = cap.read()
    finally:
        cap.release()
    if not ok:
        return None
    h, w = frame.shape[:2]
    frame = cv2.resize(frame, (lebar, int(h * lebar / w)))
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    cv2.imwrite(out_path, frame)
    return out_path


def _record(rid, lokasi, banjar, kamera, waktu, per_jenis, status, thumbnail=None):
    per_jenis = {k: int(v) for k, v in per_jenis.items()}
    urut = sorted(((k, v) for k, v in per_jenis.items() if v > 0), key=lambda x: -x[1])
    return {
        "id": rid, "lokasi": lokasi, "banjar": banjar, "kamera": kamera, "waktu": waktu,
        "jumlah_sampah": int(sum(per_jenis.values())),
        "per_jenis": per_jenis,
        "jenis_teks": ", ".join(k for k, _ in urut) or "-",
        "status": {k: status[k] for k in ("kode", "label", "prioritas", "warna_hex", "keterangan")},
        "thumbnail": thumbnail,
    }


def catat_pemeriksaan(hasil, lokasi, banjar="", kamera="", waktu=None, path="riwayat.json"):
    """Simpan hasil analyze_video() sebagai 1 baris riwayat pemeriksaan. Return record-nya."""
    s = hasil["summary"]
    data = muat_riwayat(path)
    rid = max([r["id"] for r in data], default=0) + 1
    waktu = waktu or datetime.now().strftime(FORMAT_WAKTU)
    thumb = None
    if hasil.get("video") and os.path.exists(hasil["video"]):
        thumb = simpan_thumbnail(hasil["video"], os.path.join(
            os.path.dirname(os.path.abspath(path)), "thumb", f"pemeriksaan_{rid}.jpg"))
    rec = _record(rid, lokasi, banjar, kamera, waktu, s["jumlah_per_jenis"], s["status"], thumb)
    data.append(rec)
    _simpan(path, data)
    return rec


# ---------------------------------------------------------------- halaman Dashboard
def terbaru_per_lokasi(riwayat):
    """Pemeriksaan paling baru dari tiap lokasi."""
    t = {}
    for r in sorted(riwayat, key=lambda r: r["waktu"]):
        t[r["lokasi"]] = r
    return list(t.values())


def ringkasan_dashboard(riwayat):
    """Data halaman 'Kondisi kebersihan desa' (kartu angka, lokasi per tingkat, prioritas, jenis terbanyak)."""
    terbaru = terbaru_per_lokasi(riwayat)
    jenis = Counter()
    for r in terbaru:
        jenis.update(r["per_jenis"])

    per_tingkat = []
    for kode in STATUS_URUT:
        per_tingkat.append({"kode": kode, "label": STATUS[kode]["label"], "warna_hex": STATUS[kode]["hex"],
                            "jumlah_lokasi": sum(1 for r in terbaru if r["status"]["kode"] == kode)})

    urut = sorted(terbaru, key=lambda r: (-STATUS_URUT.index(r["status"]["kode"]), -r["jumlah_sampah"]))
    prioritas = [{"urutan": i, "lokasi": r["lokasi"], "banjar": r["banjar"], "status": r["status"]["label"],
                  "kode": r["status"]["kode"], "warna_hex": r["status"]["warna_hex"],
                  "jumlah_sampah": r["jumlah_sampah"]} for i, r in enumerate(urut, 1)]

    return {
        "lokasi_dipantau": len(terbaru),
        "total_sampah": int(sum(r["jumlah_sampah"] for r in terbaru)),
        "jenis_terbanyak": jenis.most_common(1)[0][0] if jenis else "-",
        "lokasi_prioritas_tinggi": sum(1 for r in terbaru if r["status"]["kode"] == "sangat_perlu_dibersihkan"),
        "pembaruan_terakhir": max((r["waktu"] for r in terbaru), default="-"),
        "lokasi_per_tingkat": per_tingkat,
        "jenis_sampah_terbanyak": [{"jenis": k, "jumlah": int(v)} for k, v in jenis.most_common()],
        "prioritas_pembersihan": prioritas,
    }


# ---------------------------------------------------------------- halaman Riwayat
def daftar_lokasi(riwayat):
    return sorted({r["lokasi"] for r in riwayat})


def riwayat_lokasi(riwayat, lokasi):
    """Data halaman 'Riwayat pemeriksaan' untuk satu lokasi: grafik batang + daftar semua pemeriksaan."""
    rows = sorted((r for r in riwayat if r["lokasi"] == lokasi), key=lambda r: r["waktu"])
    return {
        "lokasi": lokasi,
        "banjar": rows[-1]["banjar"] if rows else "",
        "kondisi_terbaru": rows[-1]["status"] if rows else None,
        "grafik": [{"waktu": r["waktu"], "jumlah_sampah": r["jumlah_sampah"],
                    "status": r["status"]["label"], "warna_hex": r["status"]["warna_hex"]} for r in rows],
        "daftar": [{"waktu": r["waktu"], "thumbnail": r["thumbnail"], "jenis": r["jenis_teks"],
                    "jumlah_sampah": r["jumlah_sampah"], "status": r["status"]["label"],
                    "warna_hex": r["status"]["warna_hex"], "prioritas": r["status"]["prioritas"]}
                   for r in reversed(rows)],
    }


# ---------------------------------------------------------------- halaman Pantau CCTV
def kartu_html(hasil, lokasi="", kamera=""):
    """Panel kanan 'Pantau CCTV': kondisi kebersihan, prioritas penanganan, total sampah, jenis dan jumlah."""
    s = hasil["summary"]
    st = s["status"]
    baris = "".join(
        f'<div style="display:flex;justify-content:space-between;padding:7px 0;border-bottom:1px solid #e3e7dc">'
        f'<span>{k}</span><b>{v}</b></div>' for k, v in s["jumlah_per_jenis"].items())
    return f"""
<div style="font-family:sans-serif;max-width:360px;background:#f1f4ee;padding:14px;border-radius:14px">
  <div style="font-size:12px;color:#666">{kamera} {('· ' + lokasi) if lokasi else ''}</div>
  <div style="background:#fff;border-radius:12px;padding:14px;margin:8px 0">
    <div style="font-size:12px;color:#666">Kondisi kebersihan</div>
    <div style="display:inline-block;margin:6px 0;padding:6px 14px;border-radius:999px;font-weight:700;
         color:{st['warna_hex']};background:{st['warna_hex']}22">● {st['label']}</div>
    <div style="display:flex;justify-content:space-between;font-size:13px"><span>Prioritas penanganan</span><b>{st['prioritas']}</b></div>
    <div style="display:flex;justify-content:space-between;font-size:13px"><span>Total sampah terdeteksi</span><b>{st['jumlah_sampah']}</b></div>
    <div style="font-size:12px;color:#666;margin-top:6px">{st['keterangan']}</div>
  </div>
  <div style="background:#fff;border-radius:12px;padding:14px">
    <div style="font-weight:700;margin-bottom:4px">Jenis dan jumlah</div>{baris}
  </div>
</div>"""


# ---------------------------------------------------------------- data contoh (untuk uji dashboard tanpa video)
def data_contoh(path, batas=(5, 15, 28)):
    """Isi riwayat.json dengan data tiruan mirip mockup, supaya tim dashboard bisa langsung mencoba."""
    def bagi(n):
        a, b, c = round(n * .40), round(n * .30), round(n * .15)
        return {"Botol plastik": a, "Kantong plastik": b, "Kaleng": c, "Kardus": max(0, n - a - b - c)}

    banjar = "Banjar Contoh"
    contoh = [
        ("Sungai", "CCTV 3", "2026-10-12 07:30", 41), ("Tempat wisata", "CCTV 4", "2026-10-12 08:00", 33),
        ("Jalan desa", "CCTV 1", "2026-10-12 08:30", 19), ("Pantai", "CCTV 5", "2026-10-12 09:00", 3),
        ("Balai banjar", "CCTV 6", "2026-10-12 09:30", 10), ("Pasar", "CCTV 7", "2026-10-12 10:00", 14),
        ("Lapangan", "CCTV 8", "2026-10-12 10:30", 6),
        ("Taman", "CCTV 2", "2026-09-14 08:00", 12), ("Taman", "CCTV 2", "2026-09-21 08:00", 24),
        ("Taman", "CCTV 2", "2026-09-26 08:00", 36), ("Taman", "CCTV 2", "2026-10-05 08:00", 4),
        ("Taman", "CCTV 2", "2026-10-12 08:00", 8),
    ]
    data = []
    for i, (lok, cam, waktu, n) in enumerate(contoh, 1):
        data.append(_record(i, lok, banjar, cam, waktu, bagi(n), klasifikasi_kebersihan(n, batas)))
    _simpan(path, data)
    return data


Writing /content/dashboard_data.py


In [11]:
import importlib
sys.path.insert(0, BASE)
import detector; importlib.reload(detector)
from detector import analyze_video, print_summary
import dashboard_data; importlib.reload(dashboard_data)
from dashboard_data import (catat_pemeriksaan, muat_riwayat, ringkasan_dashboard,
                            riwayat_lokasi, daftar_lokasi, kartu_html, data_contoh)
print("detector.py & dashboard_data.py siap dipakai")

detector.py & dashboard_data.py siap dipakai


## 4. Video CCTV: upload dan ekstrak frame

Boleh upload **beberapa video sekaligus**. Makin beragam (siang/malam, sudut kamera, jenis sampah), makin bagus. Target minimal: **100+ frame** yang berisi sampah.

In [12]:
VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".mpeg", ".mpg")
up = files.upload()
VIDEO_PATHS = [f"{BASE}/{n}" for n in up if n.lower().endswith(VIDEO_EXT)]
if not VIDEO_PATHS:
    raise ValueError("Tidak ada video yang di-upload.")

shutil.rmtree(RAW_DIR, ignore_errors=True); RAW_DIR.mkdir(parents=True)

for vp in VIDEO_PATHS:
    slug = re.sub(r"\W+", "", Path(vp).stem)[:20] or "vid"
    cap = cv2.VideoCapture(vp)
    if not cap.isOpened():
        print("GAGAL dibuka:", vp); continue
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    step = max(1, int(round(fps * FRAME_EVERY_SEC)))
    prev, saved, idx = None, 0, 0
    while saved < MAX_FRAMES:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % step == 0:
            small = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (96, 54)).astype(np.float32)
            if prev is None or np.abs(small - prev).mean() > MIN_DIFF:
                h, w = frame.shape[:2]
                if max(h, w) > MAX_SIDE:
                    s = MAX_SIDE / max(h, w)
                    frame = cv2.resize(frame, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
                cv2.imwrite(str(RAW_DIR / f"{slug}_{idx:07d}.jpg"), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
                saved += 1
            prev = small
        idx += 1
    cap.release()
    print(f"{Path(vp).name}: {W}x{H} | {fps:.1f} FPS | {n_total/fps:.1f} dtk | frame tersimpan: {saved}")

print("\nTotal frame:", len(list(RAW_DIR.glob("*.jpg"))), "| Folder:", RAW_DIR)

Saving vid0102.mp4 to vid0102.mp4
vid0102.mp4: 588x360 | 30.0 FPS | 14.4 dtk | frame tersimpan: 27

Total frame: 27 | Folder: /content/cctv_frames


## 5. Pseudo-label (model TACO melabeli frame CCTV)

In [13]:
base = YOLO(BEST_TACO)
frames = sorted(glob.glob(f"{RAW_DIR}/*.jpg"))
LAB_ALL = Path(f"{BASE}/cctv_labels_auto")
shutil.rmtree(LAB_ALL, ignore_errors=True); LAB_ALL.mkdir(parents=True)

n_box, per_cls = 0, Counter()
for i in range(0, len(frames), 16):
    for r in base.predict(frames[i:i + 16], conf=PSEUDO_CONF, imgsz=IMGSZ, device=DEVICE, verbose=False):
        stem = Path(r.path).stem
        xywhn = r.boxes.xywhn.cpu().numpy()
        cls   = r.boxes.cls.int().cpu().numpy()
        lines = [f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, (x, y, w, h) in zip(cls, xywhn)]
        (LAB_ALL / f"{stem}.txt").write_text("\n".join(lines))
        n_box += len(lines); per_cls.update(cls.tolist())

print(f"Auto-label selesai: {len(frames)} frame, {n_box} box")
for i, n in enumerate(CLASS_NAMES):
    print(f"  {n:<16}: {per_cls[i]}")

PKG = f"{BASE}/cctv_review.zip"
with zipfile.ZipFile(PKG, "w", zipfile.ZIP_DEFLATED) as z:
    for f in frames:
        z.write(f, f"images/{os.path.basename(f)}")
    for f in LAB_ALL.glob("*.txt"):
        z.write(f, f"labels/{f.name}")
    z.writestr("classes.txt", "\n".join(CLASS_NAMES) + "\n")
print("Paket review:", PKG)
files.download(PKG)

Auto-label selesai: 27 frame, 96 box
  botol_plastik   : 47
  kantong_plastik : 40
  kaleng          : 2
  kardus          : 7
Paket review: /content/cctv_review.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Koreksi label (manual, wajib)

1. Buka `cctv_review.zip` di **Roboflow / CVAT / LabelImg**.
2. Hapus box yang salah, tambah box yang terlewat (model TACO masih lemah, jadi **banyak sampah pasti terlewat**).
3. Export format **YOLO** dan upload zip hasilnya di sel berikut.

Urutan ID kelas harus tetap sesuai `CLASS_NAMES` (lihat `classes.txt` di zip).

## 6. Fine-tune TACO + CCTV

In [14]:
USE_REVIEWED = True    # False = pakai pseudo-label apa adanya (hasil kurang bisa dipercaya)

LABEL_SRC = Path(f"{BASE}/cctv_labels_reviewed")
shutil.rmtree(LABEL_SRC, ignore_errors=True); LABEL_SRC.mkdir(parents=True)

def clean_stem(stem):
    # Roboflow menambah akhiran: f_0000030_jpg.rf.abc123 -> f_0000030
    return re.sub(r"_(jpg|jpeg|png)\.rf\..*$", "", stem)

if USE_REVIEWED:
    up = files.upload()
    zname = next(iter(up))
    ex_dir = Path(f"{BASE}/cctv_reviewed"); shutil.rmtree(ex_dir, ignore_errors=True)
    zipfile.ZipFile(zname).extractall(ex_dir)
    src_files = [t for t in glob.glob(f"{ex_dir}/**/*.txt", recursive=True)
                 if os.path.basename(t).lower() not in ("classes.txt", "labels.txt", "readme.txt")]
else:
    src_files = [str(p) for p in LAB_ALL.glob("*.txt")]

n_lab, n_box, bad = 0, 0, 0
for t in src_files:
    keep = []
    for line in Path(t).read_text().splitlines():
        p = line.split()
        if len(p) == 5 and p[0].isdigit() and int(p[0]) < len(CLASS_NAMES):
            keep.append(line.strip())
        elif line.strip():
            bad += 1
    (LABEL_SRC / f"{clean_stem(Path(t).stem)}.txt").write_text("\n".join(keep))
    n_lab += 1; n_box += len(keep)

print(f"File label: {n_lab} | box: {n_box} | baris tidak valid dibuang: {bad}")

# Cek: apakah label benar-benar dikoreksi dibanding pseudo-label?
changed = 0
for p in LABEL_SRC.glob("*.txt"):
    a = LAB_ALL / p.name
    n_auto = len([l for l in a.read_text().splitlines() if l.strip()]) if a.exists() else 0
    n_rev  = len([l for l in p.read_text().splitlines() if l.strip()])
    changed += int(n_auto != n_rev)
print(f"Frame yang jumlah box-nya berubah setelah koreksi: {changed}/{n_lab}")
if USE_REVIEWED and changed == 0:
    print("PERINGATAN: tidak ada frame yang berubah. Kalau label belum dikoreksi, model hanya belajar dari kesalahannya sendiri.")
unmatched = [p.stem for p in RAW_DIR.glob("*.jpg") if not (LABEL_SRC / f"{p.stem}.txt").exists()]
if unmatched:
    print(f"{len(unmatched)} frame tidak punya file label (dilewati), contoh:", unmatched[:3])

Saving cctv_review_dikoreksi.zip to cctv_review_dikoreksi.zip
File label: 42 | box: 628 | baris tidak valid dibuang: 0
Frame yang jumlah box-nya berubah setelah koreksi: 42/42


In [15]:
# Bangun dataset CCTV: hanya frame yang punya file label (file kosong = frame tanpa sampah / background)
stems = sorted(p.stem for p in RAW_DIR.glob("*.jpg") if (LABEL_SRC / f"{p.stem}.txt").exists())
print("Frame berlabel:", len(stems), "dari", len(list(RAW_DIR.glob('*.jpg'))))

FINAL_MODEL = BEST_TACO
FORCE_FINETUNE = False   # True = tetap fine-tune walau label belum dikoreksi
if len(stems) < 20:
    print("Frame berlabel < 20: fine-tune dilewati (data terlalu sedikit). Memakai model TACO.")
elif USE_REVIEWED and changed == 0 and not FORCE_FINETUNE:
    print("Label CCTV belum dikoreksi (0 frame berubah): fine-tune dilewati supaya model tidak belajar dari "
          "tebakannya sendiri. Koreksi label lalu jalankan ulang dari sel upload label, atau set FORCE_FINETUNE = True.")
else:
    # Split berurutan PER VIDEO (20% frame terakhir tiap video jadi val): frame bertetangga hampir sama,
    # split acak akan bocor ke val.
    by_video = defaultdict(list)
    for s in stems:
        by_video[s.rsplit("_", 1)[0]].append(s)
    split_stems = {"train": [], "val": []}
    for vid, sl in by_video.items():
        n_val = max(1, int(round(len(sl) * 0.2)))
        split_stems["train"] += sl[:-n_val]; split_stems["val"] += sl[-n_val:]

    shutil.rmtree(CCTV_DIR, ignore_errors=True)
    for split, sl in split_stems.items():
        (CCTV_DIR / "images" / split).mkdir(parents=True)
        (CCTV_DIR / "labels" / split).mkdir(parents=True)
        reps = CCTV_REPEAT if split == "train" else 1
        for s in sl:
            for r in range(reps):
                tag = f"_r{r}" if reps > 1 else ""
                shutil.copy2(RAW_DIR / f"{s}.jpg", CCTV_DIR / "images" / split / f"{s}{tag}.jpg")
                shutil.copy2(LABEL_SRC / f"{s}.txt", CCTV_DIR / "labels" / split / f"{s}{tag}.txt")

    names_map = {i: n for i, n in enumerate(CLASS_NAMES)}
    MIX_YAML  = f"{BASE}/data_mix.yaml"
    CCTV_YAML = f"{BASE}/data_cctv.yaml"
    Path(MIX_YAML).write_text(yaml.safe_dump({
        "path": BASE,
        "train": [str(TACO_DIR / "images/train"), str(CCTV_DIR / "images/train")],
        "val":   [str(TACO_DIR / "images/val"),   str(CCTV_DIR / "images/val")],
        "names": names_map}, sort_keys=False))
    Path(CCTV_YAML).write_text(yaml.safe_dump({
        "path": str(CCTV_DIR), "train": "images/train", "val": "images/val", "names": names_map}, sort_keys=False))
    print(f"CCTV train: {len(split_stems['train'])} frame x{CCTV_REPEAT} | val: {len(split_stems['val'])} frame")

    ft = YOLO(BEST_TACO)
    ft.train(
        data=MIX_YAML, epochs=FT_EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=2,
        project=RUNS_DIR, name=FT_NAME, exist_ok=True, seed=SEED, patience=PATIENCE,
        freeze=10, lr0=0.002,      # backbone dibekukan + lr kecil: data CCTV sedikit, cegah overfitting
        plots=True, amp=True, verbose=True,
    )
    BEST_FT = str(ft.trainer.best)

    # Bandingkan model TACO vs fine-tune pada val TACO dan val CCTV (terpisah)
    res = {}
    for nm, w in (("TACO", BEST_TACO), ("TACO+CCTV", BEST_FT)):
        res[nm] = {"taco_val": eval_model(w, DATA_YAML, f"{nm}_taco")[0],
                   "cctv_val": eval_model(w, CCTV_YAML, f"{nm}_cctv")[0]}
    print(f"\n{'Model':<12}{'mAP50 TACO val':>16}{'mAP50 CCTV val':>16}")
    for nm, v in res.items():
        print(f"{nm:<12}{v['taco_val']:>16.3f}{v['cctv_val']:>16.3f}")

    better_cctv = res["TACO+CCTV"]["cctv_val"] > res["TACO"]["cctv_val"]
    taco_ok = res["TACO+CCTV"]["taco_val"] >= res["TACO"]["taco_val"] - 0.03
    FINAL_MODEL = BEST_FT if (better_cctv and taco_ok) else BEST_TACO
    print("\nModel final dipilih:", "TACO+CCTV (fine-tune)" if FINAL_MODEL == BEST_FT else "TACO saja (fine-tune tidak membantu)")

print("Model final:", FINAL_MODEL)
try:
    shutil.copy2(FINAL_MODEL, f"{DRIVE_OUT}/yolo26s_final_best.pt")
    print("Backup Drive:", f"{DRIVE_OUT}/yolo26s_final_best.pt")
except Exception as e:
    print("Backup Drive dilewati:", repr(e))

Frame berlabel: 27 dari 27
CCTV train: 22 frame x5 | val: 5 frame
Ultralytics 8.4.213 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_mix.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/yolo26s_taco/weights/best.pt, momentum=0.9

## 7. Deteksi + tracking + hitung sampah

- **Unik**: jumlah ID track berbeda (track < `MIN_HITS` frame dibuang sebagai flicker)
- **Menetap**: track yang terlihat ≥ `STATIC_SEC` detik dan hampir tidak bergeser (< `STATIC_MOVE` diagonal frame)
- **Status kebersihan** (4 tingkat sesuai dashboard), dihitung dari jumlah sampah (`HITUNG_DARI`) dengan ambang `BATAS_KEBERSIHAN`:
  Bersih → Cukup Bersih → Perlu Dibersihkan → Sangat Perlu Dibersihkan (prioritas penanganan Rendah / Rendah / Sedang / Tinggi).
- Setiap analisis disimpan ke **riwayat pemeriksaan** (`riwayat.json`), yang jadi sumber data semua halaman dashboard.

- **Menetap** dinilai setelah gerak kamera dibuang (`COMPENSATE_CAMERA`). Kalau sampah yang jelas diam masih tidak terhitung, lihat kolom `bergeser_pct` di tabel tracking lalu naikkan `STATIC_MOVE` sedikit di atas angka itu.


In [16]:
OUT_DIR = f"{BASE}/output"
hasil = analyze_video(VIDEO_PATHS[0], FINAL_MODEL, out_dir=OUT_DIR, tag="cctv",
                      conf=CONF, imgsz=IMGSZ, tracker=TRACKER, min_hits=MIN_HITS,
                      static_sec=STATIC_SEC, static_move=STATIC_MOVE, out_width=OUT_WIDTH, device=DEVICE,
                      hitung_dari=HITUNG_DARI, batas=BATAS_KEBERSIHAN, compensate_camera=COMPENSATE_CAMERA,
                      progress=lambda i, n: print(f"  frame {i}/{n}") if i % 100 == 0 else None)
print("\n=== RINGKASAN ===")
print_summary(hasil)

# Simpan ke riwayat pemeriksaan (sumber data dashboard)
rec = catat_pemeriksaan(hasil, lokasi=LOKASI, banjar=BANJAR, kamera=KAMERA, path=RIWAYAT_JSON)
print(f"\nTersimpan di riwayat: #{rec['id']} | {rec['lokasi']} | {rec['waktu']} | {rec['status']['label']}")

# Tampilan panel kanan halaman "Pantau CCTV"
from IPython.display import HTML
display(HTML(kartu_html(hasil, lokasi=LOKASI, kamera=KAMERA)))

print("\nVideo:", hasil["video"]); print("CSV  :", hasil["csv"])
import pandas as pd
display(pd.read_csv(hasil["csv"])[["track_id", "kelas", "durasi_dtk", "bergeser_pct", "conf_rata2", "menetap"]].head(30))
display(Video(hasil["video"], embed=True, width=720))


  frame 100/433
  frame 200/433
  frame 300/433
  frame 400/433

=== RINGKASAN ===
Kelas               Unik   Menetap
botol_plastik          0         0
kantong_plastik        9         9
kaleng                 0         0
kardus                 0         0
TOTAL                  9         9

KONDISI KEBERSIHAN: CUKUP BERSIH | prioritas Rendah | 9 sampah (menetap)
Lokasi cukup bersih. Ada 9 sampah, masih aman tetapi perlu dipantau.

Tersimpan di riwayat: #1 | Taman | 2026-11-15 07:53 | Cukup Bersih



Video: /content/output/cctv_hasil.mp4
CSV  : /content/output/cctv_tracking.csv


,track_id,kelas,durasi_dtk,bergeser_pct,conf_rata2,menetap
0,1,kantong_plastik,14.43,0.15,0.950,1
1,2,kantong_plastik,14.43,0.04,0.938,1
2,3,kantong_plastik,14.43,0.09,0.944,1
3,4,kantong_plastik,14.43,0.20,0.945,1
4,5,kantong_plastik,14.43,0.14,0.936,1
5,6,kantong_plastik,14.43,0.09,0.936,1
6,7,kantong_plastik,14.43,0.05,0.946,1
7,8,kantong_plastik,14.43,0.05,0.927,1
8,9,kantong_plastik,14.43,0.05,0.932,1


## 7b. Data dashboard dari riwayat pemeriksaan

Tiga fungsi ini yang dipanggil Streamlit untuk tiga halaman: **Dashboard** (`ringkasan_dashboard`), **Riwayat** (`riwayat_lokasi`), dan **Pantau CCTV** (`kartu_html`). Set `PAKAI_DATA_CONTOH = True` untuk mengisi data tiruan mirip mockup (8 lokasi) kalau belum punya banyak video.

In [17]:
PAKAI_DATA_CONTOH = False
if PAKAI_DATA_CONTOH:
    data_contoh(RIWAYAT_JSON, batas=BATAS_KEBERSIHAN)

riwayat = muat_riwayat(RIWAYAT_JSON)
dash = ringkasan_dashboard(riwayat)

print("=== HALAMAN DASHBOARD: Kondisi kebersihan desa ===")
print(f"{dash['lokasi_dipantau']} lokasi dipantau | {dash['total_sampah']} sampah terdeteksi | "
      f"terbanyak: {dash['jenis_terbanyak']} | {dash['lokasi_prioritas_tinggi']} lokasi perhatian tinggi")
display(pd.DataFrame(dash["lokasi_per_tingkat"])[["label", "jumlah_lokasi"]])
display(pd.DataFrame(dash["jenis_sampah_terbanyak"]))
display(pd.DataFrame(dash["prioritas_pembersihan"])[["urutan", "lokasi", "banjar", "status", "jumlah_sampah"]])

if riwayat:
    lok = daftar_lokasi(riwayat)[0]
    print(f"\n=== HALAMAN RIWAYAT: {lok} ===")
    rw = riwayat_lokasi(riwayat, lok)
    display(pd.DataFrame(rw["daftar"])[["waktu", "jenis", "jumlah_sampah", "status", "prioritas"]])

with open(f"{DASH_DIR}/dashboard.json", "w", encoding="utf-8") as f:
    json.dump(dash, f, indent=2, ensure_ascii=False)
print("\nJSON dashboard:", f"{DASH_DIR}/dashboard.json")


=== HALAMAN DASHBOARD: Kondisi kebersihan desa ===
1 lokasi dipantau | 9 sampah terdeteksi | terbanyak: Kantong plastik | 0 lokasi perhatian tinggi


,label,jumlah_lokasi
0,Bersih,0
1,Cukup Bersih,1
2,Perlu Dibersihkan,0
3,Sangat Perlu Dibersihkan,0


,jenis,jumlah
0,Kantong plastik,9
1,Botol plastik,0
2,Kaleng,0
3,Kardus,0


,urutan,lokasi,banjar,status,jumlah_sampah
0,1,Taman,Banjar Contoh,Cukup Bersih,9



=== HALAMAN RIWAYAT: Taman ===


,waktu,jenis,jumlah_sampah,status,prioritas
0,2026-11-15 07:53,Kantong plastik,9,Cukup Bersih,Rendah



JSON dashboard: /content/dashboard_data/dashboard.json


## 8. Uji video baru (opsional, jalankan sesering perlu)

In [18]:
up = files.upload()
vids = [n for n in up if n.lower().endswith(VIDEO_EXT)]
if not vids:
    raise ValueError("Tidak ada video yang di-upload.")

# Ganti sesuai lokasi/kamera video yang diuji
LOKASI, BANJAR, KAMERA = "Taman", "Banjar Contoh", "CCTV 1"

h2 = analyze_video(f"{BASE}/{vids[-1]}", FINAL_MODEL, out_dir=OUT_DIR, tag="video_baru",
                   conf=CONF, imgsz=IMGSZ, tracker=TRACKER, min_hits=MIN_HITS,
                   static_sec=STATIC_SEC, static_move=STATIC_MOVE, out_width=OUT_WIDTH, device=DEVICE,
                   hitung_dari=HITUNG_DARI, batas=BATAS_KEBERSIHAN, compensate_camera=COMPENSATE_CAMERA)
print_summary(h2)
catat_pemeriksaan(h2, lokasi=LOKASI, banjar=BANJAR, kamera=KAMERA, path=RIWAYAT_JSON)
display(HTML(kartu_html(h2, lokasi=LOKASI, kamera=KAMERA)))
display(Video(h2["video"], embed=True, width=720))
files.download(h2["video"]); files.download(h2["csv"])


Saving vid0037.mp4 to vid0037.mp4
Kelas               Unik   Menetap
botol_plastik          0         0
kantong_plastik        0         0
kaleng                 0         0
kardus                 0         0
TOTAL                  0         0

KONDISI KEBERSIHAN: BERSIH | prioritas Rendah | 0 sampah (menetap)
Lokasi bersih. Hanya 0 sampah terdeteksi.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
up = files.upload()
vids = [n for n in up if n.lower().endswith(VIDEO_EXT)]
if not vids:
    raise ValueError("Tidak ada video yang di-upload.")

# Ganti sesuai lokasi/kamera video yang diuji
LOKASI, BANJAR, KAMERA = "Taman", "Banjar Contoh", "CCTV 1"

h2 = analyze_video(f"{BASE}/{vids[-1]}", FINAL_MODEL, out_dir=OUT_DIR, tag="video_baru",
                   conf=CONF, imgsz=IMGSZ, tracker=TRACKER, min_hits=MIN_HITS,
                   static_sec=STATIC_SEC, static_move=STATIC_MOVE, out_width=OUT_WIDTH, device=DEVICE,
                   hitung_dari=HITUNG_DARI, batas=BATAS_KEBERSIHAN, compensate_camera=COMPENSATE_CAMERA)
print_summary(h2)
catat_pemeriksaan(h2, lokasi=LOKASI, banjar=BANJAR, kamera=KAMERA, path=RIWAYAT_JSON)
display(HTML(kartu_html(h2, lokasi=LOKASI, kamera=KAMERA)))
display(Video(h2["video"], embed=True, width=720))
files.download(h2["video"]); files.download(h2["csv"])


Saving vid0292.mp4 to vid0292.mp4
Kelas               Unik   Menetap
botol_plastik          9         5
kantong_plastik       19         6
kaleng                 4         3
kardus                 4         2
TOTAL                 36        16

KONDISI KEBERSIHAN: PERLU DIBERSIHKAN | prioritas Sedang | 16 sampah (menetap)
Lokasi perlu dibersihkan. Ada 16 sampah, jadwalkan pembersihan.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Serahkan ke tim (Orang 3 / Streamlit)

Dua file ini yang dibutuhkan dashboard: **`detector.py`** dan **bobot model `.pt`**.
Pemakaian di Streamlit: `from detector import analyze_video` lalu `analyze_video(path_video, "best.pt", out_dir="output")`.
File yang dibutuhkan dashboard: **`detector.py`**, **`dashboard_data.py`**, dan **bobot model `.pt`**. Data riwayat ada di `riwayat.json`.

```python
from detector import analyze_video
from dashboard_data import catat_pemeriksaan, muat_riwayat, ringkasan_dashboard, riwayat_lokasi, kartu_html

hasil = analyze_video(path_video, "best.pt", out_dir="output")
catat_pemeriksaan(hasil, lokasi="Taman", banjar="Banjar X", kamera="CCTV 1", path="riwayat.json")
dash = ringkasan_dashboard(muat_riwayat("riwayat.json"))   # halaman Dashboard
```


In [21]:
shutil.copy2(FINAL_MODEL, f"{BASE}/best.pt")
files.download(f"{BASE}/detector.py")
files.download(f"{BASE}/best.pt")
files.download(f"{BASE}/dashboard_data.py")
if os.path.exists(RIWAYAT_JSON):
    files.download(RIWAYAT_JSON)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>